# 📂 Document Loaders & Text Splitters — Hands-On Examples

> **Module 03 | Submodule 05 | Practical Notebook**
>
> Sections:
> 1. The Document object
> 2. PyPDFLoader — PDF loading
> 3. Directory loader — load all PDFs in a folder
> 4. WebBaseLoader — web pages
> 5. CSVLoader & WikipediaLoader
> 6. RecursiveCharacterTextSplitter
> 7. Comparing chunk sizes
> 8. MarkdownHeaderTextSplitter
> 9. Full RAG ingestion pipeline

In [ ]:
!pip install langchain langchain-openai langchain-community \
             langchain-text-splitters pypdf beautifulsoup4 \
             wikipedia python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "module-03-loaders-splitters"
print("Ready!")

---
## 1️⃣ The Document Object

In [ ]:
from langchain_core.documents import Document

# Create documents manually
docs = [
    Document(
        page_content="LangChain is a framework for building LLM applications.",
        metadata={"source": "intro.md", "topic": "langchain", "level": "beginner"}
    ),
    Document(
        page_content="LangGraph extends LangChain with stateful, graph-based agent orchestration.",
        metadata={"source": "intro.md", "topic": "langgraph", "level": "intermediate"}
    ),
    Document(
        page_content="LangSmith is the observability platform for LangChain applications.",
        metadata={"source": "ecosystem.md", "topic": "langsmith", "level": "intermediate"}
    ),
]

# Inspect a Document
doc = docs[0]
print("Type:",        type(doc))
print("Content:",     doc.page_content)
print("Metadata:",    doc.metadata)
print("Content len:", len(doc.page_content))

---
## 2️⃣ PyPDFLoader — Loading PDFs

In [ ]:
# First, create a simple test PDF using reportlab (or download one)
# If you don't have a PDF, we'll create a sample
try:
    from reportlab.pdfgen import canvas
    import tempfile, os

    # Create a sample PDF for testing
    tmp_pdf = "/tmp/sample_doc.pdf"
    c = canvas.Canvas(tmp_pdf)
    c.setFont("Helvetica", 12)
    c.drawString(50, 750, "Introduction to LangChain")
    c.drawString(50, 720, "LangChain is a framework for building LLM-powered applications.")
    c.drawString(50, 700, "It provides tools for prompting, chaining, and memory management.")
    c.showPage()
    c.drawString(50, 750, "Core Components")
    c.drawString(50, 720, "Models: LLMs and Chat Models from various providers.")
    c.drawString(50, 700, "Prompts: Templates for structuring LLM inputs.")
    c.drawString(50, 680, "Parsers: Output parsers for structured responses.")
    c.save()
    print("Sample PDF created at:", tmp_pdf)
    PDF_PATH = tmp_pdf
except ImportError:
    print("reportlab not installed. Replace PDF_PATH with your own PDF file.")
    PDF_PATH = "your_document.pdf"  # ← Replace with your PDF path

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(PDF_PATH)
pdf_docs = loader.load()

print(f"Pages loaded:  {len(pdf_docs)}")
print(f"\nPage 0 metadata: {pdf_docs[0].metadata}")
print(f"Page 0 content:  {pdf_docs[0].page_content[:200]}")
if len(pdf_docs) > 1:
    print(f"\nPage 1 content: {pdf_docs[1].page_content[:200]}")

In [ ]:
# Lazy loading — memory-efficient for large PDFs
loader = PyPDFLoader(PDF_PATH)

print("Lazy loading pages:")
for doc in loader.lazy_load():
    print(f"  Page {doc.metadata['page']}: {len(doc.page_content)} chars")

---
## 3️⃣ Directory Loader — Load Multiple Files

In [ ]:
import os, tempfile

# Create a temp directory with some text files
tmpdir = tempfile.mkdtemp()
files = {
    "langchain.txt": "LangChain is a framework for building LLM applications with composable components.",
    "langgraph.txt": "LangGraph extends LangChain with graph-based stateful agent workflows.",
    "langsmith.txt": "LangSmith provides tracing, evaluation, and monitoring for LangChain apps.",
}
for name, content in files.items():
    with open(os.path.join(tmpdir, name), "w") as f:
        f.write(content * 5)  # Repeat for longer content

print(f"Created {len(files)} files in {tmpdir}")

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    path=tmpdir,
    glob="**/*.txt",
    loader_cls=TextLoader,
    show_progress=True
)
dir_docs = loader.load()

print(f"Total documents: {len(dir_docs)}")
for doc in dir_docs:
    source = os.path.basename(doc.metadata['source'])
    print(f"  {source}: {len(doc.page_content)} chars")

---
## 4️⃣ WebBaseLoader — Web Pages

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
import bs4

# Load a web page
loader = WebBaseLoader(
    web_paths=["https://python.langchain.com/docs/introduction/"],
    bs_kwargs={"parse_only": bs4.SoupStrainer(class_("theme-doc-markdown"))} if False else {}
)

web_docs = loader.load()
print(f"Loaded {len(web_docs)} page(s)")
print(f"URL: {web_docs[0].metadata['source']}")
print(f"Title: {web_docs[0].metadata.get('title', 'N/A')}")
print(f"Content length: {len(web_docs[0].page_content)} chars")
print(f"\nFirst 500 chars:")
print(web_docs[0].page_content[:500])

In [ ]:
# Load multiple URLs at once
multi_loader = WebBaseLoader([
    "https://python.langchain.com/docs/introduction/",
    "https://python.langchain.com/docs/concepts/",
])
multi_docs = multi_loader.load()

print(f"Loaded {len(multi_docs)} pages")
for doc in multi_docs:
    print(f"  {doc.metadata['source']}: {len(doc.page_content)} chars")

---
## 5️⃣ CSV & Wikipedia Loaders

In [ ]:
import csv, tempfile

# Create a sample CSV
csv_file = tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False)
writer = csv.DictWriter(csv_file, fieldnames=['product', 'description', 'price'])
writer.writeheader()
writer.writerows([
    {'product': 'LangChain Pro', 'description': 'Enterprise-grade LLM orchestration framework', 'price': '99'},
    {'product': 'LangSmith', 'description': 'Observability and testing for LLM apps', 'price': '49'},
    {'product': 'LangFlow', 'description': 'Visual workflow builder for LLM pipelines', 'price': '0'},
])
csv_file.close()

from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(
    file_path=csv_file.name,
    source_column="product"   # This column becomes the source in metadata
)
csv_docs = loader.load()

print(f"Rows loaded: {len(csv_docs)}")
for doc in csv_docs:
    print(f"\n--- {doc.metadata['source']} ---")
    print(doc.page_content)

In [ ]:
from langchain_community.document_loaders import WikipediaLoader

loader = WikipediaLoader(query="LangChain AI framework", load_max_docs=1)
wiki_docs = loader.load()

print(f"Articles: {len(wiki_docs)}")
print(f"Title: {wiki_docs[0].metadata.get('title')}")
print(f"Content length: {len(wiki_docs[0].page_content)} chars")
print(f"Preview: {wiki_docs[0].page_content[:400]}")

---
## 6️⃣ RecursiveCharacterTextSplitter

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

long_text = """
LangChain is a framework for developing applications powered by language models.

It provides a standard interface for models, prompts, and output parsers.
The main abstractions in LangChain are:

Models: Support for LLMs, Chat Models, and Embedding Models from dozens of providers
including OpenAI, Anthropic, Google, and local models via Ollama.

Prompts: PromptTemplates and ChatPromptTemplates for building reusable, parameterized
prompts. These support variables, partial filling, and few-shot examples.

Output Parsers: Convert raw LLM output into structured types. Options include
StrOutputParser for strings, JsonOutputParser for dicts, and PydanticOutputParser
for fully typed and validated Pydantic models.

Chains: Compose components using LCEL (LangChain Expression Language) with the
pipe | operator. Chains support streaming, batching, async, and full LangSmith tracing.

Retrievers: Components that fetch relevant documents from a vector store. Used in
RAG (Retrieval-Augmented Generation) pipelines to ground LLM responses in real data.
"""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.create_documents([long_text])

print(f"Original: {len(long_text)} chars")
print(f"Chunks:   {len(chunks)}")
print()
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content.strip())
    print()

In [ ]:
# Split documents while preserving metadata
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)
chunks = splitter.split_documents(web_docs)

print(f"Web docs: {len(web_docs)} → Chunks: {len(chunks)}")
print(f"\nSample chunk metadata: {chunks[0].metadata}")
print(f"Source preserved: {chunks[0].metadata.get('source', '✅')}")

---
## 7️⃣ Comparing Chunk Sizes

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load a document to chunk
source_text = web_docs[0].page_content if web_docs else long_text

configs = [
    {"chunk_size": 200,  "chunk_overlap": 20},
    {"chunk_size": 500,  "chunk_overlap": 50},
    {"chunk_size": 1000, "chunk_overlap": 200},
    {"chunk_size": 2000, "chunk_overlap": 400},
]

print(f"Source length: {len(source_text)} chars\n")
print(f"{'chunk_size':>12} {'overlap':>10} {'n_chunks':>10} {'avg_len':>10} {'min':>8} {'max':>8}")
print("-" * 65)

for cfg in configs:
    s = RecursiveCharacterTextSplitter(**cfg)
    c = s.create_documents([source_text])
    sizes = [len(x.page_content) for x in c]
    print(f"{cfg['chunk_size']:>12} {cfg['chunk_overlap']:>10} {len(c):>10} {sum(sizes)//len(sizes):>10} {min(sizes):>8} {max(sizes):>8}")

---
## 8️⃣ MarkdownHeaderTextSplitter

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_doc = """
# LangChain Framework

LangChain is a powerful framework for building LLM applications.

## Installation

Install LangChain with pip:
```bash
pip install langchain langchain-openai
```

### OpenAI Setup

Set your API key in the environment: OPENAI_API_KEY=sk-...

### Anthropic Setup

Set your API key: ANTHROPIC_API_KEY=sk-ant-...

## Core Concepts

LangChain is built around the Runnable interface.

### LCEL

LCEL stands for LangChain Expression Language. It uses | to compose Runnables.

### Memory

Memory lets agents remember past conversations and context.
"""

headers_to_split_on = [
    ("#",   "h1"),
    ("##",  "h2"),
    ("###", "h3"),
]

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False
)
md_chunks = splitter.split_text(markdown_doc)

print(f"Markdown chunks: {len(md_chunks)}\n")
for chunk in md_chunks:
    print(f"Headers: {chunk.metadata}")
    print(f"Content: {chunk.page_content.strip()[:100]}")
    print()

---
## 9️⃣ Full RAG Ingestion Pipeline

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from datetime import datetime

print("=" * 50)
print("RAG Ingestion Pipeline")
print("=" * 50)

# STEP 1: Load
print("\n📥 Step 1: Loading documents...")
loader = WebBaseLoader(["https://python.langchain.com/docs/introduction/"])
raw_docs = loader.load()
print(f"  Loaded: {len(raw_docs)} document(s)")
print(f"  Total chars: {sum(len(d.page_content) for d in raw_docs):,}")

# STEP 2: Enrich metadata
print("\n🏷️  Step 2: Enriching metadata...")
for doc in raw_docs:
    doc.metadata["ingested_at"] = datetime.now().isoformat()
    doc.metadata["domain"]      = "langchain-docs"
    doc.metadata["version"]     = "v0.3"
print(f"  Added: ingested_at, domain, version")

# STEP 3: Split
print("\n✂️  Step 3: Splitting into chunks...")
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    length_function=len,
)
chunks = splitter.split_documents(raw_docs)
print(f"  Chunks: {len(chunks)}")
sizes  = [len(c.page_content) for c in chunks]
print(f"  Avg size: {sum(sizes)//len(sizes)} chars")
print(f"  Min: {min(sizes)} | Max: {max(sizes)} chars")

# STEP 4: Quality check
print("\n🔍 Step 4: Quality check...")
empty = [c for c in chunks if len(c.page_content.strip()) < 50]
print(f"  Empty/tiny chunks: {len(empty)} → {'❌ fix needed' if empty else '✅ none'}")
print(f"  All have source: {'✅' if all('source' in c.metadata for c in chunks) else '❌'}")

# Inspect a sample chunk
import random
sample = random.choice(chunks)
print(f"\n📄 Sample chunk:")
print(f"  Metadata: {sample.metadata}")
print(f"  Content: {sample.page_content.strip()[:200]}...")

print("\n✅ Ready for embedding & vector store (next module)!")
print(f"   → {len(chunks)} chunks prepared")

---
## ✅ Summary

| Component | Purpose | Output |
|---|---|---|
| `PyPDFLoader` | Load PDFs | 1 `Document` per page |
| `WebBaseLoader` | Load web pages | 1 `Document` per URL |
| `CSVLoader` | Load CSVs | 1 `Document` per row |
| `WikipediaLoader` | Load Wikipedia | 1 `Document` per article |
| `DirectoryLoader` | Load all files in folder | N `Document`s |
| `RecursiveCharacterTextSplitter` | Split by natural boundaries | List of smaller `Document`s |
| `MarkdownHeaderTextSplitter` | Split by MD headers | Metadata-enriched chunks |

**Next**: [06 — Vector Stores & Retrievers](../06_vector_stores_and_retrievers/theory.md)